# Notebook 28 — Residual Operator Phase Transitions and Multi-Scale Continuity Flow

This notebook extends the paper's **Extensions and Future Directions** section.

Notebook 27 established a static residual spectral hierarchy:

\[
	ext{continuity} ightarrow 	ext{transport} ightarrow 	ext{fragmentation}.
\]

Notebook 28 asks how that hierarchy **evolves** under perturbation, graph scaling, transport coupling, and fragmentation pressure.

Core outputs:

- `28_spectral_phase_flow.png`
- `28_operator_stability_curves.png`
- `28_phase_boundary_map.png`
- `28_spectral_persistence_landscape.png`
- `28_transition_heatmap.png`
- `28_transition_metrics.csv`
- `28_phase_flow_summary.json`


In [ ]:
# Setup
from pathlib import Path
import json, math, zipfile, warnings, subprocess, sys

try:
    import networkx as nx
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'networkx'])
    import networkx as nx

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

ROOT = Path('/content') if Path('/content').exists() else Path('.')
FIGURES_DIR = ROOT / 'figures'
RESULTS_DIR = ROOT / 'results'
EXPORTS_DIR = ROOT / 'exports'
for d in [FIGURES_DIR, RESULTS_DIR, EXPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RNG_SEED = 9423
np.random.seed(RNG_SEED)

FAMILIES = ['ring lattice', 'small world', 'Erdős–Rényi', 'scale free', 'modular clustered']
SIZES = [16, 32, 64, 128, 256]
PERTURB_LEVELS = np.linspace(0.0, 1.0, 11)
REPS = 3

print('Notebook 28 setup complete')
print('figures:', FIGURES_DIR)
print('results:', RESULTS_DIR)


## 1. Residual operator transition design

We model a perturbation parameter \(q\in[0,1]\).  Each topology family evolves under a family-specific perturbation rule:

- ring lattice: increasing rewiring pressure,
- small world: increasing shortcut/randomization pressure,
- Erdős–Rényi: increasing edge probability variance,
- scale free: increasing hub concentration pressure,
- modular clustered: increasing inter-community leakage.

The dynamic residual operator stack is represented as:

\[
\mathcal{O}_q = \mathcal{S}_q \circ \mathcal{D}_q \circ \mathcal{T}_q \circ ho_q.
\]

The goal is to track when residual structure moves through continuity, transport, and fragmentation regimes.


In [ ]:
def safe_connected(G, seed=0):
    if G.number_of_nodes() == 0:
        return G
    if nx.is_connected(G):
        return G
    comps = [list(c) for c in nx.connected_components(G)]
    rng = np.random.default_rng(seed)
    for a, b in zip(comps[:-1], comps[1:]):
        G.add_edge(int(rng.choice(a)), int(rng.choice(b)))
    return G


def make_modular_graph(n, leakage=0.05, modules=4, seed=0):
    rng = np.random.default_rng(seed)
    sizes = [n // modules] * modules
    for i in range(n % modules):
        sizes[i] += 1
    G = nx.Graph()
    G.add_nodes_from(range(n))
    starts = np.cumsum([0] + sizes)
    groups = [list(range(starts[i], starts[i+1])) for i in range(modules)]
    p_in = max(0.25, 0.70 - 0.25 * leakage)
    p_out = min(0.35, 0.01 + 0.22 * leakage)
    for group in groups:
        for i, u in enumerate(group):
            for v in group[i+1:]:
                if rng.random() < p_in:
                    G.add_edge(u, v)
    for a in range(modules):
        for b in range(a+1, modules):
            for u in groups[a]:
                for v in groups[b]:
                    if rng.random() < p_out:
                        G.add_edge(u, v)
    return safe_connected(G, seed=seed)


def make_graph(family, n, q, seed=0):
    rng = np.random.default_rng(seed)
    if family == 'ring lattice':
        k = max(4, min(n-1, int(round(4 + 4*q))))
        if k % 2 == 1:
            k += 1
        p = 0.03 + 0.45*q
        return safe_connected(nx.watts_strogatz_graph(n, k, p, seed=seed), seed=seed)
    if family == 'small world':
        k = max(4, min(n-1, int(round(6 + 2*q))))
        if k % 2 == 1:
            k += 1
        p = 0.10 + 0.65*q
        return safe_connected(nx.watts_strogatz_graph(n, k, p, seed=seed), seed=seed)
    if family == 'Erdős–Rényi':
        p = min(0.55, 0.10 + 0.35*q + 2.0/n)
        return safe_connected(nx.erdos_renyi_graph(n, p, seed=seed), seed=seed)
    if family == 'scale free':
        m = max(1, min(n-1, int(round(1 + 3*q))))
        G = nx.barabasi_albert_graph(n, m, seed=seed)
        # hub reinforcement under q
        hub = max(dict(G.degree()).items(), key=lambda kv: kv[1])[0]
        extra = int(q * n * 0.25)
        for _ in range(extra):
            v = int(rng.integers(0, n))
            if v != hub:
                G.add_edge(hub, v)
        return safe_connected(G, seed=seed)
    if family == 'modular clustered':
        return make_modular_graph(n, leakage=q, modules=4, seed=seed)
    raise ValueError(f'Unknown family: {family}')


def laplacian_spectrum(G):
    A = nx.to_numpy_array(G, dtype=float)
    deg = A.sum(axis=1)
    L = np.diag(deg) - A
    vals = np.linalg.eigvalsh(L)
    return np.sort(np.maximum(vals, 0.0))


def spectral_entropy(vals, eps=1e-12):
    vals = np.asarray(vals, dtype=float)
    total = vals.sum()
    if total <= eps:
        return 0.0
    p = vals / total
    p = p[p > eps]
    return float(-(p * np.log(p)).sum())


def graph_features(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    degrees = np.array([d for _, d in G.degree()], dtype=float)
    vals = laplacian_spectrum(G)
    gaps = np.diff(vals)
    lam2 = vals[1] if len(vals) > 1 else 0.0
    lam_max = vals[-1] if len(vals) else 0.0
    avg_deg = float(degrees.mean()) if n else 0.0
    deg_cv = float(degrees.std() / (avg_deg + 1e-9))
    clustering = float(nx.average_clustering(G))
    density = float(nx.density(G))
    tri = sum(nx.triangles(G).values()) / 3.0
    entropy = spectral_entropy(vals) / (np.log(max(n, 2)) + 1e-12)
    max_gap = float(gaps.max()) if len(gaps) else 0.0
    max_gap_idx = int(gaps.argmax()) if len(gaps) else 0

    # Operator-inspired scores, normalized to [0, 1] by smooth bounded functions.
    continuity = float(np.clip(0.45*clustering + 0.35*np.exp(-deg_cv) + 0.20*np.exp(-entropy), 0, 1))
    transport = float(np.clip(0.55*np.tanh(lam2 / (avg_deg + 1e-9) * 4.0) + 0.45*np.tanh(density*5), 0, 1))
    fragmentation = float(np.clip(0.55*np.tanh(deg_cv) + 0.45*np.tanh(max_gap / (avg_deg + 1e-9)), 0, 1))
    stability = float(np.clip(0.50*continuity + 0.35*transport - 0.45*fragmentation + 0.35, 0, 1))

    return dict(
        n=n, edges=m, density=density, avg_degree=avg_deg,
        degree_cv=deg_cv, clustering=clustering, triangles=tri,
        lambda2=float(lam2), lambda_max=float(lam_max),
        spectral_entropy_norm=entropy, max_eigengap=max_gap,
        max_eigengap_index=max_gap_idx,
        continuity_score=continuity, transport_score=transport,
        fragmentation_score=fragmentation, stability_score=stability,
    )


## 2. Run the perturbation experiment

This cell generates topology-family graphs across scale, perturbation level, and repeated seeds.

It exports a tidy metrics table as:

```text
results/28_transition_metrics.csv
```


In [ ]:
rows = []
for family in FAMILIES:
    for n in SIZES:
        for q in PERTURB_LEVELS:
            for rep in range(REPS):
                seed = RNG_SEED + 1000*FAMILIES.index(family) + 10*n + rep + int(round(q*100))
                G = make_graph(family, n, float(q), seed=seed)
                feats = graph_features(G)
                rows.append(dict(family=family, N=n, perturbation=float(q), rep=rep, **feats))

metrics = pd.DataFrame(rows)
metrics_path = RESULTS_DIR / '28_transition_metrics.csv'
metrics.to_csv(metrics_path, index=False)
print('Wrote:', metrics_path)
print(metrics.shape)
metrics.head()


## 3. Residual spectral phase-flow embedding

We embed the operator metrics into a residual phase-flow plane using PCA.

This gives an interpretable view of how topology families move through perturbation pressure.


In [ ]:
feature_cols = [
    'density', 'avg_degree', 'degree_cv', 'clustering', 'lambda2', 'lambda_max',
    'spectral_entropy_norm', 'max_eigengap', 'continuity_score', 'transport_score',
    'fragmentation_score', 'stability_score'
]
X = metrics[feature_cols].fillna(0.0).values
Xs = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=RNG_SEED)
Z = pca.fit_transform(Xs)
metrics['phase_pc1'] = Z[:, 0]
metrics['phase_pc2'] = Z[:, 1]

phase_path = RESULTS_DIR / '28_phase_embedding.csv'
metrics.to_csv(phase_path, index=False)
print('Explained variance:', pca.explained_variance_ratio_)
print('Wrote:', phase_path)


In [ ]:
# Figure 1: spectral phase flow
fig, ax = plt.subplots(figsize=(11, 7))

for family in FAMILIES:
    sub = metrics[metrics.family == family].groupby('perturbation')[['phase_pc1','phase_pc2']].mean().reset_index()
    ax.plot(sub.phase_pc1, sub.phase_pc2, marker='o', linewidth=2, label=family)
    for i in range(len(sub)-1):
        ax.annotate('', xy=(sub.phase_pc1.iloc[i+1], sub.phase_pc2.iloc[i+1]),
                    xytext=(sub.phase_pc1.iloc[i], sub.phase_pc2.iloc[i]),
                    arrowprops=dict(arrowstyle='->', lw=1.2, alpha=0.45))
    ax.text(sub.phase_pc1.iloc[-1], sub.phase_pc2.iloc[-1], 'q=1', fontsize=9)

ax.axhline(0, linestyle='--', alpha=0.4)
ax.axvline(0, linestyle='--', alpha=0.4)
ax.set_title('Residual spectral phase flow under perturbation')
ax.set_xlabel('operator phase PC1')
ax.set_ylabel('operator phase PC2')
ax.legend(loc='best')
ax.grid(True, alpha=0.25)
fig.tight_layout()
path = FIGURES_DIR / '28_spectral_phase_flow.png'
fig.savefig(path, dpi=180)
plt.show()
print('Wrote:', path)


## 4. Operator stability and transition curves

We now track the transition from continuity into transport and fragmentation regimes.

A simple operator stability score is computed from continuity, transport, and fragmentation measurements:

\[
R_q = 0.50 C_q + 0.35 T_q - 0.45 F_q + 0.35.
\]

This score is not a theorem; it is a reproducible diagnostic for comparing regime motion across topology families.


In [ ]:
summary = metrics.groupby(['family','perturbation']).agg(
    continuity=('continuity_score','mean'),
    transport=('transport_score','mean'),
    fragmentation=('fragmentation_score','mean'),
    stability=('stability_score','mean'),
    eigengap=('max_eigengap','mean'),
    entropy=('spectral_entropy_norm','mean')
).reset_index()

summary_path = RESULTS_DIR / '28_phase_flow_summary.csv'
summary.to_csv(summary_path, index=False)
print('Wrote:', summary_path)
summary.head()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
for family in FAMILIES:
    sub = summary[summary.family == family]
    ax.plot(sub.perturbation, sub.stability, marker='o', linewidth=2, label=family)

ax.set_title('Operator stability under perturbation')
ax.set_xlabel('perturbation level q')
ax.set_ylabel('mean residual operator stability')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.25)
ax.legend(loc='best')
fig.tight_layout()
path = FIGURES_DIR / '28_operator_stability_curves.png'
fig.savefig(path, dpi=180)
plt.show()
print('Wrote:', path)


In [ ]:
# Figure 3: transition heatmap by family and perturbation
pivot = summary.pivot(index='family', columns='perturbation', values='stability').loc[FAMILIES]
fig, ax = plt.subplots(figsize=(12, 4.8))
im = ax.imshow(pivot.values, aspect='auto', vmin=0, vmax=1)
ax.set_title('Residual operator stability transition heatmap')
ax.set_xlabel('perturbation level q')
ax.set_ylabel('topology family')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{c:.1f}' for c in pivot.columns], rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i,j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, label='operator stability')
fig.tight_layout()
path = FIGURES_DIR / '28_transition_heatmap.png'
fig.savefig(path, dpi=180)
plt.show()
print('Wrote:', path)


## 5. Phase boundary map

The paper defined continuity, transport, and fragmentation regimes.  Here we sketch a diagnostic phase boundary map using two aggregate pressures:

- transport coupling,
- fragmentation pressure.

The color field shows continuity stability.


In [ ]:
# Analytic phase boundary map based on transport coupling and fragmentation pressure.
t_grid = np.linspace(0, 1, 121)
f_grid = np.linspace(0, 1, 121)
TT, FF = np.meshgrid(t_grid, f_grid)
CC = np.clip(0.75 + 0.35*TT - 0.85*FF - 0.20*TT*FF, 0, 1)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.contourf(TT, FF, CC, levels=20)
cs = ax.contour(TT, FF, CC, levels=[0.33, 0.66], linestyles='--', linewidths=1.5)
ax.clabel(cs, inline=True, fontsize=9)
ax.set_title('Residual operator phase boundary map')
ax.set_xlabel('transport coupling')
ax.set_ylabel('fragmentation pressure')
ax.text(0.08, 0.15, 'continuity', fontsize=11)
ax.text(0.42, 0.45, 'transport', fontsize=11)
ax.text(0.65, 0.78, 'fragmentation', fontsize=11)
fig.colorbar(im, ax=ax, label='continuity stability')
fig.tight_layout()
path = FIGURES_DIR / '28_phase_boundary_map.png'
fig.savefig(path, dpi=180)
plt.show()
print('Wrote:', path)


## 6. Spectral persistence landscape

This figure extends the cumulative eigengap hierarchy from Notebook 27.

We track normalized eigengap persistence as a surface over perturbation level and spectral mode.


In [ ]:
# Build spectral persistence from average spectra at N=128 for each family and q.
spec_rows = []
N_REF = 128
MAX_MODES = 18
for family in FAMILIES:
    for q in PERTURB_LEVELS:
        spectra = []
        for rep in range(REPS):
            seed = RNG_SEED + 9999 + 1000*FAMILIES.index(family) + rep + int(round(q*100))
            G = make_graph(family, N_REF, float(q), seed=seed)
            vals = laplacian_spectrum(G)
            gaps = np.diff(vals[:MAX_MODES+1])
            if gaps.max() > 0:
                gaps = gaps / gaps.max()
            spectra.append(gaps[:MAX_MODES])
        mean_gaps = np.mean(np.vstack(spectra), axis=0)
        for k, g in enumerate(mean_gaps):
            spec_rows.append(dict(family=family, perturbation=float(q), mode=k, gap_persistence=float(g)))

spec_df = pd.DataFrame(spec_rows)
spec_path = RESULTS_DIR / '28_spectral_persistence.csv'
spec_df.to_csv(spec_path, index=False)
print('Wrote:', spec_path)


In [ ]:
# Plot persistence landscape averaged across families.
land = spec_df.groupby(['perturbation','mode'])['gap_persistence'].mean().reset_index()
P = land.pivot(index='perturbation', columns='mode', values='gap_persistence')

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(P.values, aspect='auto', origin='lower', vmin=0, vmax=1,
               extent=[P.columns.min(), P.columns.max(), P.index.min(), P.index.max()])
ax.set_title('Spectral persistence landscape')
ax.set_xlabel('spectral mode k')
ax.set_ylabel('perturbation level q')
fig.colorbar(im, ax=ax, label='normalized eigengap persistence')
fig.tight_layout()
path = FIGURES_DIR / '28_spectral_persistence_landscape.png'
fig.savefig(path, dpi=180)
plt.show()
print('Wrote:', path)


## 7. Phase-flow summary JSON

Export a compact machine-readable summary for README / paper / docs integration.


In [ ]:
family_summary = {}
for family in FAMILIES:
    sub = summary[summary.family == family].sort_values('perturbation')
    initial = float(sub.stability.iloc[0])
    final = float(sub.stability.iloc[-1])
    min_idx = int(sub.stability.argmin())
    family_summary[family] = {
        'initial_stability': initial,
        'final_stability': final,
        'stability_delta': final - initial,
        'minimum_stability': float(sub.stability.iloc[min_idx]),
        'minimum_stability_q': float(sub.perturbation.iloc[min_idx]),
        'mean_continuity': float(sub.continuity.mean()),
        'mean_transport': float(sub.transport.mean()),
        'mean_fragmentation': float(sub.fragmentation.mean()),
    }

summary_json = {
    'notebook': '28_residual_operator_phase_transitions.ipynb',
    'seed': RNG_SEED,
    'families': FAMILIES,
    'sizes': SIZES,
    'perturbation_levels': [float(x) for x in PERTURB_LEVELS],
    'family_summary': family_summary,
    'created_outputs': {
        'figures': sorted([p.name for p in FIGURES_DIR.glob('28_*.png')]),
        'results': sorted([p.name for p in RESULTS_DIR.glob('28_*')]),
    }
}

json_path = RESULTS_DIR / '28_phase_flow_summary.json'
json_path.write_text(json.dumps(summary_json, indent=2))
print('Wrote:', json_path)
print(json.dumps(family_summary, indent=2)[:1200])


## 8. Export package with optional Colab download

This cell zips Notebook 28 figures, results, and manifest.


In [ ]:
manifest = {
    'notebook': '28_residual_operator_phase_transitions.ipynb',
    'created_outputs': {
        'figures': sorted([p.name for p in FIGURES_DIR.glob('28_*.png')]),
        'results': sorted([p.name for p in RESULTS_DIR.glob('28_*')]),
    },
    'notes': 'Notebook 28 initializes extensions and future directions from the residual manifold operator paper.'
}

manifest_path = EXPORTS_DIR / '28_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

zip_path = EXPORTS_DIR / '28_residual_operator_phase_transitions_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in FIGURES_DIR.glob('28_*.png'):
        z.write(p, arcname=f'figures/{p.name}')
    for p in RESULTS_DIR.glob('28_*'):
        z.write(p, arcname=f'results/{p.name}')
    z.write(manifest_path, arcname='exports/28_manifest.json')

print('Wrote:', zip_path)
print('Zip size MB:', zip_path.stat().st_size / 1e6)

# Optional Colab download
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Colab download skipped. Download manually from:', zip_path)


## Notebook 28 conclusion

Notebook 28 initializes the extension path from the paper.  It treats the residual operator hierarchy as dynamic:

\[
	ext{continuity} ightarrow 	ext{transport} ightarrow 	ext{fragmentation}
\]

and tracks how this hierarchy evolves under perturbation, scale, and spectral persistence.

The next notebook could specialize one branch:

- weighted transport operators,
- temporal topology evolution,
- multi-layer residual manifolds,
- continuity-energy minimization,
- curvature-informed transport flow.
